# 🚀 Knowledge Distillation: Fine-tune viT5-Base làm Query Rewriter Y Khoa

**Mục tiêu:** Fine-tune mô hình Seq2Seq mạnh mẽ `VietAI/vit5-base` (~737M params) trên dataset được chưng cất nhằm tối ưu hóa từ khóa tìm kiếm (Query Rewriting) chuẩn xác tuyệt đối cho hệ thống Medical RAG.

### 📊 Thông tin tập dữ liệu (Dataset Summary):
- **Tập huấn luyện (Train Set):** 2,661 cặp `(source -> target)`
- **Tập kiểm thử (Test Set):** 296 cặp `(source -> target)` để báo cáo đánh giá độc lập.
- **Môi trường khuyến nghị:** Kaggle GPU T4 x2 hoặc P100 / V100.

In [1]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q transformers datasets evaluate rouge_score accelerate peft torch tqdm pandas numpy sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [2]:
import os
import json
import torch
import evaluate
import numpy as np
import pandas as pd
import sentencepiece as spm
from huggingface_hub import hf_hub_download
from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

GPU Available: True
Device: Tesla T4


## 2. Chuẩn bị & Nạp Dữ liệu (Train / Test Split)
Lưu ý: Đảm bảo 2 file `train.jsonl` và `test.jsonl` đã được tải lên Kaggle Input hoặc nằm trong cùng thư mục làm việc.

In [3]:
train_path = "train.jsonl"
test_path = "test.jsonl"

if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train.jsonl" in files:
            train_path = os.path.join(root, "train.jsonl")
        if "test.jsonl" in files:
            test_path = os.path.join(root, "test.jsonl")

print("Train file:", train_path)
print("Test file:", test_path)

data_files = {"train": train_path, "test": test_path}
dataset = load_dataset("json", data_files=data_files)
print(dataset)

Train file: /kaggle/input/datasets/tu4nhoang/clean-query-rewrite/train.jsonl
Test file: /kaggle/input/datasets/tu4nhoang/clean-query-rewrite/test.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source', 'target'],
        num_rows: 2661
    })
    test: Dataset({
        features: ['source', 'target'],
        num_rows: 296
    })
})


## 3. Khởi tạo Tokenizer chuẩn xác 36,000 từ (Khắc phục triệt để lỗi sinh chữ rỗng)
Do thư viện `transformers` bản mới thay đổi cách nạp từ điển T5, chúng ta dùng `SentencePieceProcessor` đọc trực tiếp bộ từ điển 36,000 từ chuẩn tiếng Việt từ `spiece.model` và truyền vào `T5Tokenizer(vocab=vocab_list)`.

In [4]:
MODEL_NAME = "VietAI/vit5-base"
vocab_file = hf_hub_download(repo_id=MODEL_NAME, filename="spiece.model")

# Đọc trọn bộ từ điển 36,000 từ chuẩn của SentencePiece
sp = spm.SentencePieceProcessor(model_file=vocab_file)
vocab_list = [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())]
tokenizer = T5Tokenizer(vocab=vocab_list)
print(f"✅ Đã nạp thành công bộ từ điển chuẩn với {tokenizer.vocab_size:,} từ khóa!")

PREFIX = "tối ưu từ khóa tìm kiếm y khoa: "
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 48

def preprocess_function(examples):
    inputs = [PREFIX + src for src in examples["source"]]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    
    labels = tokenizer(text_target=examples["target"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)
print("✅ Tiền xử lý dữ liệu thành công!")

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

✅ Đã nạp thành công bộ từ điển chuẩn với 36,000 từ khóa!


Map:   0%|          | 0/2661 [00:00<?, ? examples/s]

Map:   0%|          | 0/296 [00:00<?, ? examples/s]

✅ Tiền xử lý dữ liệu thành công!


## 4. Tải Mô hình VietAI/vit5-base & Thiết lập Trainer (Siêu tốc & Chống OOM)

In [5]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
rouge = evaluate.load("rouge")

# In thông số mô hình
total_params = sum(p.numel() for p in model.parameters())
print(f"📌 Tên mô hình: {MODEL_NAME}")
print(f"🔥 Tổng số tham số: {total_params:,} (~{total_params / 1e6:.1f}M params)")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # 🌟 LỌC BỎ SỐ ÂM (< 0 như -100 do GPU padding) TRƯỚC KHI ĐƯA CHO BỘ GIẢI MÃ RUST
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    labels = np.where(labels < 0, tokenizer.pad_token_id, labels)
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    return {k: round(v * 100, 2) for k, v in result.items()}

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-vi-medical-rewriter",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4, # LR 3e-4 chuẩn mực tối ưu cho T5-Base
    optim="adafactor",   # Bộ tối ưu Adafactor chuyên dụng cho T5 giải phóng 5GB VRAM
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,  # Tắt checkpointing để train siêu nhanh 5 phút
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    generation_max_length=48,  # Khóa độ dài sinh 48 từ, đánh giá nhanh 10 giây
    fp16=False,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

📌 Tên mô hình: VietAI/vit5-base
🔥 Tổng số tham số: 309,116,160 (~309.1M params)


## 5. Bắt đầu Huấn luyện (Fine-Tuning)

In [6]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,4.808100,1.249804,47.250000,30.110000,41.780000,41.830000,3780.070000
2,2.167387,1.110804,59.880000,40.670000,52.590000,52.660000,2453.720000
3,1.442800,1.001026,65.200000,46.860000,56.770000,56.800000,2458.110000
4,0.975287,1.045866,66.730000,47.880000,58.320000,58.360000,2653.040000
5,0.621682,1.112372,68.090000,49.830000,59.630000,59.680000,2476.010000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=420, training_loss=1.849071827388945, metrics={'train_runtime': 491.8878, 'train_samples_per_second': 27.049, 'train_steps_per_second': 0.854, 'total_flos': 2305910897021952.0, 'train_loss': 1.849071827388945, 'epoch': 5.0})

## 6. Báo Cáo Đánh Giá ROUGE Score Trên Tập Test (296 mẫu)

In [7]:
test_results = trainer.evaluate(max_length=MAX_TARGET_LENGTH)
print("=== BÁO CÁO KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (296 mẫu) ===")
for k, v in test_results.items():
    print(f"{k}: {v}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


=== BÁO CÁO KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (296 mẫu) ===
eval_loss: 1.1123716831207275
eval_rouge1: 68.09
eval_rouge2: 49.83
eval_rougeL: 59.63
eval_rougeLsum: 59.68
eval_gen_len: 2476.01
eval_runtime: 12.4273
eval_samples_per_second: 23.818
eval_steps_per_second: 0.805
epoch: 5.0


## 7. Kiểm chứng Trực quan (Sách Đối chiếu Ground Truth vs Prediction)

In [8]:
test_samples = dataset["test"].shuffle(seed=42).select(range(15))

results_table = []
for row in test_samples:
    src = row["source"]
    gt = row["target"]
    
    input_text = PREFIX + src
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(input_ids, max_new_tokens=MAX_TARGET_LENGTH, num_beams=2, repetition_penalty=1.5, no_repeat_ngram_size=2)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    results_table.append({
        "Câu hỏi gốc (Source)": src[:80] + ("..." if len(src) > 80 else ""),
        "Từ khóa chuẩn (Ground Truth)": gt,
        "T5 Dự đoán (Prediction)": pred
    })

df_report = pd.DataFrame(results_table)
display(df_report)

,Câu hỏi gốc (Source),Từ khóa chuẩn (Ground Truth),T5 Dự đoán (Prediction)
0,Tại sao việc sử dụng các thuốc cảm ứng enzym g...,marvelon tương tác rifampicin carbamazepin thu...,Marvelon chuyển sang biện pháp tránh thai khôn...
1,Quá trình chuyển hóa và thải trừ của gadoliniu...,gadolinium chuyển hóa thải trừ thời gian bán thải,thời gian bán thải gadolinium chuyển hóa
2,So với quan điểm 'phòng bệnh tim mạch' của Tây...,dầu hạt cải tính vị quy kinh đông y tây y,dầu hạt cải chống viêm kháng virus bệnh mãn tính
3,Ethotoin ức chế các xung thần kinh trong vỏ nã...,Ethotoin ức chế xung thần kinh vỏ não vận động...,Ethotoin ức chế xung thần kinh vỏ não vận động...
4,Phân tích sự khác biệt về chỉ định sử dụng Hya...,hyaluronic acid tiêm nội khớp nhỏ mắt biến chứ...,hyaluronic acid công thức tiêm nội khớp nhỏ mắ...
5,Tôi đang điều trị suy thận nặng với độ thanh t...,sancefur 35mg suy thận nặng độ thanh thải crea...,sancefur 35mg suy thận nặng độ thanh thải crea...
6,Bệnh nhân của tôi đang dùng Glyxambi nhưng đổi...,Glyxambi empagliflozin linagliptin uống buổi t...,glyxambi tương tác thuốc buổi tối thường lệ
7,"Trong quá trình chuyển hóa ở gan, chất chuyển ...",chất chuyển hóa olanzapin gan hoạt tính dược lý,olanzapin chất chuyển hóa gan chống chỉ định p...
8,Một bệnh nhân đang dùng Alumag-S liều cao kéo ...,Alumag-S nhôm hydroxyd magnesi hydroxyd dùng k...,Alumag-S liều cao kéo dài loét dạ dày suy dinh...
9,Cơ chế diệt khuẩn của Tobramycin liên quan đến...,tobramycin cơ chế diệt khuẩn tiểu đơn vị ribosom,cơ chế Tobramycin diệt khuẩn tổng hợp ribosom


## 8. Đóng gói & Tải Mô hình Hoàn thiện

In [9]:
FINAL_SAVE_PATH = "/kaggle/working/t5_query_rewriter"
model.save_pretrained(FINAL_SAVE_PATH)
tokenizer.save_pretrained(FINAL_SAVE_PATH)
print(f"✅ Đã lưu mô hình thành công tại: {FINAL_SAVE_PATH}")

!cd /kaggle/working && zip -q -r t5_query_rewriter.zip t5_query_rewriter
print("🎁 File tải về đã sẵn sàng: /kaggle/working/t5_query_rewriter.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Đã lưu mô hình thành công tại: /kaggle/working/t5_query_rewriter
🎁 File tải về đã sẵn sàng: /kaggle/working/t5_query_rewriter.zip
